# NDBI021: Explore trained SAE movie codes

Requires only `numpy`, `pandas`, and Jupyter. Uses portable `sae_codes.npz` and `sae_movies.csv` exported by EasyStudy at initialization. No Flask, EasyStudy, or unpickling.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Edit to point to your exported folder; alternatively copy both files beside this notebook.
ARTIFACT_DIR = Path('.')
ARTIFACT_DIR = Path(r'D:\NDBI021_easystudy_extension\server\cache\fastcompare\kp-exI8XW9_K5vS9tcxtne1Hu-2nmjI-\sae_artifacts')
with np.load(ARTIFACT_DIR / 'sae_codes.npz', allow_pickle=False) as data:
    z = data['codes'].astype(np.float32)
    popularity = data['popularity']
movies = pd.read_csv(ARTIFACT_DIR / 'sae_movies.csv')
assert z.ndim == 2 and len(movies) == len(popularity) == z.shape[0]
assert np.array_equal(movies['item_index'].to_numpy(), np.arange(z.shape[0]))
assert np.allclose(movies['popularity'].to_numpy(), popularity)
print('SAE codes:', z.shape, '| Nonzero fraction:', np.count_nonzero(z) / z.size)
display(movies.head())

SAE codes: (1930, 256) | Nonzero fraction: 0.0625


,item_index,item_id,title,genres,popularity
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7034
1,1,2,Jumanji (1995),Adventure|Children|Fantasy,2328
2,2,6,Heat (1995),Action|Crime|Thriller,3452
3,3,10,GoldenEye (1995),Action|Adventure|Thriller,1476
4,4,16,Casino (1995),Crime|Drama,2964


## Select a movie by title substring
Case-insensitive literal substring; when several match, choose by `item_index`.

In [2]:
def find_movies(substring, n=30):
    found = movies[movies.title.str.contains(substring, case=False, regex=False, na=False)]
    display(found.sort_values('popularity', ascending=False).head(n))
    return found

def select_movie(substring, item_index=None):
    found = movies[movies.title.str.contains(substring, case=False, regex=False, na=False)]
    if item_index is not None:
        found = found[found.item_index == item_index]
    if found.empty:
        raise ValueError('No matching movie')
    if len(found) > 1:
        print('Multiple matches; choosing most popular. Pass item_index to select a specific match.')
    return int(found.sort_values('popularity', ascending=False).iloc[0].item_index)

find_movies('matrix')

,item_index,item_id,title,genres,popularity
170,170,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,11465
358,358,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3998
376,376,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3088
446,446,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,1187
1614,1614,263009,The Matrix Resurrections (2021),Action|Adventure|Sci-Fi,144


,item_index,item_id,title,genres,popularity
170,170,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,11465
358,358,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3998
376,376,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3088
446,446,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,1187
1614,1614,263009,The Matrix Resurrections (2021),Action|Adventure|Sci-Fi,144


## Top activating movies for a neuron
Show strongest **positive** and **negative** signed activations separately.

In [3]:
def top_activations(neuron, n=10):
    if not 0 <= neuron < z.shape[1]:
        raise IndexError('Invalid neuron index')
    activations = z[:, neuron]
    for label, candidates, descending in [
        ('Positive', np.flatnonzero(activations > 0), True),
        ('Negative', np.flatnonzero(activations < 0), False),
    ]:
        order = np.argsort(-activations[candidates] if descending else activations[candidates], kind='stable')
        indices = candidates[order[:n]]
        result = movies.iloc[indices].copy()
        result['activation'] = activations[indices]
        print(f'{label} activations for neuron {neuron}')
        display(result[['item_index', 'title', 'activation', 'popularity']])

top_activations(0, n=10)

Positive activations for neuron 0


,item_index,title,activation,popularity
1144,1144,Captain Marvel (2018),0.719035,956
1411,1411,Ant-Man and the Wasp (2018),0.608428,1254
1577,1577,Black Widow (2021),0.583001,304
1220,1220,Shazam! (2019),0.563046,657
1435,1435,Venom (2018),0.558346,688
1480,1480,Spider-Man: Far from Home (2019),0.552682,1744
1370,1370,Jumanji: Welcome to the Jungle (2017),0.544088,1193
1143,1143,Black Panther (2017),0.521748,2595
1401,1401,Jurassic World: Fallen Kingdom (2018),0.521414,330
1501,1501,Zombieland: Double Tap (2019),0.518812,333


Negative activations for neuron 0


,item_index,title,activation,popularity


In [4]:
top_activations(6, n=5)

Positive activations for neuron 6


,item_index,title,activation,popularity


Negative activations for neuron 6


,item_index,title,activation,popularity
1700,1700,Plane (2023),-0.621481,24
1868,1868,Death's Roulette (2023),-0.597413,1
1702,1702,The Devil Conspiracy (2023),-0.586854,1
1910,1910,No Hard Feelings (2023),-0.569588,6
1887,1887,Asteroid City (2023),-0.539299,55


## Similar movies in sparse-code space
Cosine similarity uses the full signed SAE code. Zero-code movies are excluded.

In [5]:
norms = np.linalg.norm(z, axis=1)
unit = np.divide(z, norms[:, None], out=np.zeros_like(z), where=norms[:, None] > 0)

def similar_movies(title_substring, n=15, item_index=None):
    idx = select_movie(title_substring, item_index)
    print('Query:', movies.iloc[idx].title)
    if norms[idx] == 0:
        print('Zero SAE code; cosine similarity undefined.')
        return
    scores = unit @ unit[idx]
    scores[idx] = -np.inf
    scores[norms == 0] = -np.inf
    nearest = np.argsort(-scores, kind='stable')[:n]
    nearest = nearest[np.isfinite(scores[nearest])]
    result = movies.iloc[nearest].copy()
    result['cosine_similarity'] = scores[nearest]
    result['shared_active_neurons'] = np.count_nonzero((z[nearest] != 0) & (z[idx] != 0), axis=1)
    display(result[['item_index', 'title', 'cosine_similarity', 'shared_active_neurons', 'popularity']])
    return result

similar_movies('Interstellar')

Query: Interstellar (2014)


,item_index,title,cosine_similarity,shared_active_neurons,popularity
786,786,Inception (2010),0.779015,11,11984
1174,1174,The Martian (2015),0.765602,11,7734
962,962,Django Unchained (2012),0.716784,10,8700
1107,1107,Ex Machina (2015),0.688038,9,6492
1032,1032,"Wolf of Wall Street, The (2013)",0.687722,10,7261
1011,1011,Gravity (2013),0.638861,9,4924
1070,1070,Edge of Tomorrow (2014),0.631179,8,6501
936,936,Looper (2012),0.629441,9,4631
896,896,"Dark Knight Rises, The (2012)",0.627139,8,8333
1287,1287,Arrival (2016),0.619987,8,6380


,item_index,item_id,title,genres,popularity,cosine_similarity,shared_active_neurons
786,786,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,11984,0.779015,11
1174,1174,134130,The Martian (2015),Adventure|Drama|Sci-Fi,7734,0.765602,11
962,962,99114,Django Unchained (2012),Action|Drama|Western,8700,0.716784,10
1107,1107,115713,Ex Machina (2015),Drama|Sci-Fi|Thriller,6492,0.688038,9
1032,1032,106782,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama,7261,0.687722,10
1011,1011,104841,Gravity (2013),Action|Sci-Fi|IMAX,4924,0.638861,9
1070,1070,111759,Edge of Tomorrow (2014),Action|Sci-Fi|IMAX,6501,0.631179,8
936,936,96610,Looper (2012),Action|Crime|Sci-Fi,4631,0.629441,9
896,896,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,8333,0.627139,8
1287,1287,164179,Arrival (2016),Sci-Fi,6380,0.619987,8


## Movie popularity
Popularity is the count of training interactions, **not** overall movie popularity.

In [6]:
def movie_popularity(title_substring, item_index=None):
    idx = select_movie(title_substring, item_index)
    row = movies.iloc[idx]
    rank = 1 + int((popularity > popularity[idx]).sum())
    print(f'{row.title} | MovieLens ID {row.item_id} | {int(popularity[idx]):,} interactions | rank {rank}/{len(movies)}')
    return row

movie_popularity('Interstellar')

Interstellar (2014) | MovieLens ID 109487 | 9,059 interactions | rank 11/1930


item_index                   1048
item_id                    109487
title         Interstellar (2014)
genres                Sci-Fi|IMAX
popularity                   9059
Name: 1048, dtype: object

In [7]:
def bidirectional_neurons(z):
    positive_counts = np.count_nonzero(z > 0, axis=0)
    negative_counts = np.count_nonzero(z < 0, axis=0)

    neuron_ids = np.flatnonzero(
        (positive_counts > 3) & (negative_counts > 3)
    )

    result = pd.DataFrame({
        "neuron_id": neuron_ids,
        "positive_count": positive_counts[neuron_ids],
        "negative_count": negative_counts[neuron_ids],
    })

    print(f"Bidirectional neurons: {len(result)} / {z.shape[1]}")
    display(result)

    return result


bidirectional = bidirectional_neurons(z)

Bidirectional neurons: 7 / 256


,neuron_id,positive_count,negative_count
0,11,12,74
1,28,82,4
2,83,17,27
3,93,76,6
4,195,8,69
5,227,48,9
6,230,5,58


## Suggested explorations
- Try several neurons. Are positive and negative examples semantically coherent?
- Are the closest movies also in the same genres?
- How does popularity affect which examples would be suitable for preference elicitation?